# 红利低波原始指数编制

本 notebook 从项目根目录运行，演示一次选样快照和一个固定成分区间的价格指数、全收益指数计算。日期调度、自动调仓、交易成本和绩效评价暂不在本节处理。

In [ ]:
from pathlib import Path

import pandas as pd

from pyquant.data import load_dataset
from pyquant.io import load_config
from strategies.cross_sectional.dividend_low_vol.components import (
    calculate_dividend_low_vol_index,
    select_dividend_low_vol_constituents,
)

## 日期参数

`as_of_date` 是选样信息截止日，`effective_date` 是指数区间生效日，两者刻意分离，便于未来由调仓器传入。

In [ ]:
as_of_date = pd.Timestamp("2024-11-29")
effective_date = pd.Timestamp("2024-12-02")
end_date = pd.Timestamp("2025-05-30")
history_start = as_of_date - pd.DateOffset(years=4)

## 通用数据集读取

行情明确使用不复权口径；分红、查询覆盖和股本数据均通过数据集目录读取。

In [ ]:
strategy_dir = Path("strategies/cross_sectional/dividend_low_vol")
config = load_config(strategy_dir / "config.yaml")
price = load_dataset(
    "stock_daily",
    start=history_start.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    adjustment="none",
)
dividends = load_dataset("dividend")
dividend_queries = load_dataset("dividend_queries")
shares = load_dataset("stock_profit_quarterly")

## 单次选样快照

In [ ]:
constituents = select_dividend_low_vol_constituents(
    price,
    dividends,
    dividend_queries,
    shares,
    as_of_date,
    config,
)
constituents[[
    "price_date",
    "avg_dividend_yield_3y",
    "dividend_yield_rank",
    "volatility_240d",
    "volatility_rank",
    "weight",
]].sort_values("weight", ascending=False)

## 单个调仓区间指数

首段两类指数均从 1000 点起算。未来拼接调仓区间时，将上一段生效日收盘点位分别作为下一段的两个基点，并删除重复边界行。

In [ ]:
index_segment = calculate_dividend_low_vol_index(
    price,
    dividends,
    dividend_queries,
    constituents,
    effective_date,
    end_date,
)
index_segment.tail()

In [ ]:
index_segment[["price_index", "total_return_index"]].plot(
    figsize=(12, 5),
    title="红利低波价格指数与全收益指数",
)